# Backpropagation from scratch

**Sierra Janson** · Lab for Human Mathematics · spring 2024, revised 2026 with Claude!

This notebook derives the gradients of a small neural network **by hand**, implements
them with nothing but NumPy, and then **verifies them against PyTorch's autograd**.
It starts with a two-neuron network and generalises to $n$ hidden neurons.

It also documents two pitfalls I ran into in the original 2024 version of this
notebook, with a demonstration of each:

1. **The symmetry trap.** If every weight starts at the same value, every hidden neuron
   gets the same gradient and they stay identical forever, so an $n$-neuron network acts like a 1-neuron one.
2. **A learning rate too large for the problem.** Some runs ended in overflow and `nan`
   weights because each gradient step overshot further than the last.

## 1. A network with two hidden neurons

```
            w1 ──▶ (z1) ── w3
          ╱                  ╲
   (x) ──                      ──▶ (y0)
          ╲                  ╱
            w2 ──▶ (z2) ── w4
```

Every neuron uses the **leaky ReLU** activation, with a small slope $\varepsilon > 0$ for negative inputs:

$$
\Gamma(u) = \begin{cases} \varepsilon u & u < 0 \\ u & u \ge 0 \end{cases}
\qquad
\Gamma'(u) = \begin{cases} \varepsilon & u < 0 \\ 1 & u \ge 0 \end{cases}
$$

**Forward pass**

$$
z_1 = \Gamma(w_1 x), \qquad z_2 = \Gamma(w_2 x), \qquad
s = w_3 z_1 + w_4 z_2, \qquad y_0 = \Gamma(s)
$$

**Loss.** For a target $y$ we use $L = \tfrac12 (y_0 - y)^2$ (the $\tfrac12$ just gives a clean derivative),
averaged over the training examples.

**Backward pass.** By the chain rule:

$$
\begin{aligned}
\frac{\partial L}{\partial w_3} &= \underbrace{(y_0 - y)}_{\partial L/\partial y_0}\;\underbrace{\Gamma'(s)}_{\partial y_0/\partial s}\; z_1 &
\frac{\partial L}{\partial w_4} &= (y_0 - y)\,\Gamma'(s)\, z_2 \\[4pt]
\frac{\partial L}{\partial w_1} &= (y_0 - y)\,\Gamma'(s)\, w_3\, \Gamma'(w_1 x)\, x &
\frac{\partial L}{\partial w_2} &= (y_0 - y)\,\Gamma'(s)\, w_4\, \Gamma'(w_2 x)\, x
\end{aligned}
$$

**Gradient descent** with learning rate $\eta$: $\; w_i \leftarrow w_i - \eta\, \dfrac{\partial L}{\partial w_i}$.

In [1]:
import numpy as np
import torch

EPS = 0.01  # leaky-ReLU slope for negative inputs


def leaky_relu(u):
    return np.where(u < 0, EPS * u, u)


def leaky_relu_deriv(u):
    return np.where(u < 0, EPS, 1.0)


# Training data: learn y = 5x from five examples. Inputs 6-10 are held out.
x_train = np.arange(1, 6, dtype=float)
y_train = 5 * x_train
x_unseen = np.arange(6, 11, dtype=float)

In [2]:
def forward_2(w, x):
    w1, w2, w3, w4 = w
    z1, z2 = leaky_relu(w1 * x), leaky_relu(w2 * x)
    s = w3 * z1 + w4 * z2
    return leaky_relu(s), (z1, z2, s)


def gradients_2(w, x, y):
    # The four partial derivatives derived above, averaged over all examples.
    w1, w2, w3, w4 = w
    y0, (z1, z2, s) = forward_2(w, x)
    delta = (y0 - y) * leaky_relu_deriv(s)  # dL/ds, shared by every weight
    return np.array([
        np.mean(delta * w3 * leaky_relu_deriv(w1 * x) * x),  # dL/dw1
        np.mean(delta * w4 * leaky_relu_deriv(w2 * x) * x),  # dL/dw2
        np.mean(delta * z1),                                 # dL/dw3
        np.mean(delta * z2),                                 # dL/dw4
    ])


def loss_2(w, x, y):
    return 0.5 * np.mean((forward_2(w, x)[0] - y) ** 2)


w = np.array([0.5, 0.3, 0.4, 0.6])  # distinct starting values (see section 3 for why)
eta = 0.01
print(f"start: loss = {loss_2(w, x_train, y_train):.4f}")
for epoch in range(2000):
    w = w - eta * gradients_2(w, x_train, y_train)
print(f"end:   loss = {loss_2(w, x_train, y_train):.2e},  weights = {np.round(w, 4)}")
print("unseen inputs", x_unseen, "->", np.round(forward_2(w, x_unseen)[0], 4))

start: loss = 117.3942


end:   loss = 0.00e+00,  weights = [1.5885 1.5595 1.574  1.603 ]
unseen inputs [ 6.  7.  8.  9. 10.] -> [30. 35. 40. 45. 50.]


The network learns $y = 5x$ and also gets the unseen inputs right. That's expected here:
with no biases and positive weights, the network computes $y_0 = (w_1 w_3 + w_2 w_4)\,x$,
which is exactly a line through the origin, so training only has to make $w_1 w_3 + w_2 w_4 = 5$.

## 2. Checking the derivation against PyTorch autograd

Hand-derived gradients are easy to get subtly wrong. PyTorch computes gradients by
automatic differentiation, so if the two agree to floating-point precision, the
derivation is right. I check this at random weights, where both leaky-ReLU branches get used.

In [3]:
F = torch.nn.functional


def autograd_gradients_2(w, x, y):
    wt = torch.tensor(w, requires_grad=True)
    xt, yt = torch.tensor(x), torch.tensor(y)
    z1 = F.leaky_relu(wt[0] * xt, EPS)
    z2 = F.leaky_relu(wt[1] * xt, EPS)
    y0 = F.leaky_relu(wt[2] * z1 + wt[3] * z2, EPS)
    loss = 0.5 * ((y0 - yt) ** 2).mean()
    loss.backward()
    return wt.grad.numpy()


rng = np.random.default_rng(0)
x_check = rng.normal(0, 3, size=50)  # include negative inputs
y_check = 5 * x_check
worst = 0.0
for _ in range(100):
    w_rand = rng.normal(0, 1, size=4)
    diff = np.abs(gradients_2(w_rand, x_check, y_check) - autograd_gradients_2(w_rand, x_check, y_check))
    worst = max(worst, diff.max())
print(f"largest difference over 100 random weight settings: {worst:.1e}")

largest difference over 100 random weight settings: 1.4e-14


## 3. Generalising to $n$ hidden neurons

With $n$ hidden neurons, input weights $a_i$ and output weights $b_i$:

$$
z_i = \Gamma(a_i x), \qquad s = \sum_{i=1}^{n} b_i z_i, \qquad y_0 = \Gamma(s)
$$

and the same chain rule gives, for every $i$,

$$
\frac{\partial L}{\partial b_i} = (y_0 - y)\,\Gamma'(s)\, z_i,
\qquad
\frac{\partial L}{\partial a_i} = (y_0 - y)\,\Gamma'(s)\, b_i\, \Gamma'(a_i x)\, x .
$$

In NumPy each of these is a single vectorised expression over all neurons and all examples.

In [4]:
def forward_n(a, b, x):
    pre = np.outer(x, a)        # (examples, neurons): a_i * x
    z = leaky_relu(pre)
    s = z @ b
    return leaky_relu(s), (pre, z, s)


def gradients_n(a, b, x, y):
    y0, (pre, z, s) = forward_n(a, b, x)
    delta = ((y0 - y) * leaky_relu_deriv(s))[:, None]  # (examples, 1)
    grad_b = (delta * z).mean(axis=0)
    grad_a = (delta * b * leaky_relu_deriv(pre) * x[:, None]).mean(axis=0)
    return grad_a, grad_b


def train_n(a, b, x, y, eta=0.01, epochs=2000):
    a, b = a.copy(), b.copy()
    for _ in range(epochs):
        grad_a, grad_b = gradients_n(a, b, x, y)
        a -= eta * grad_a
        b -= eta * grad_b
    return a, b


def autograd_gradients_n(a, b, x, y):
    at = torch.tensor(a, requires_grad=True)
    bt = torch.tensor(b, requires_grad=True)
    y0 = F.leaky_relu(F.leaky_relu(torch.tensor(x)[:, None] * at, EPS) @ bt, EPS)
    (0.5 * ((y0 - torch.tensor(y)) ** 2).mean()).backward()
    return at.grad.numpy(), bt.grad.numpy()


a0, b0 = rng.normal(0, 1, 15), rng.normal(0, 1, 15)
ga, gb = gradients_n(a0, b0, x_check, y_check)
ta, tb = autograd_gradients_n(a0, b0, x_check, y_check)
print(f"n = 15 gradient check, largest difference: {max(np.abs(ga - ta).max(), np.abs(gb - tb).max()):.1e}")

n = 15 gradient check, largest difference: 8.3e-17


### The symmetry trap

The 2024 version started every weight at $0.5$. Look at the gradient formulas: if all
$a_i$ are equal and all $b_i$ are equal, then every neuron gets **exactly the same gradient**,
so after the update they're all still equal. No amount of training can separate them, and
fifteen neurons behave like one. This is why real networks start from *random* weights.

In [5]:
n = 15
a_sym, b_sym = train_n(np.full(n, 0.5), np.full(n, 0.5), x_train, y_train)
a_rnd, b_rnd = train_n(rng.normal(0, 0.5, n), rng.normal(0, 0.5, n), x_train, y_train)

print("identical start: spread of input weights after training =", np.ptp(a_sym))
print("random start:    spread of input weights after training =", round(np.ptp(a_rnd), 3))

identical start: spread of input weights after training = 0.0
random start:    spread of input weights after training = 2.201


For $y = 5x$ a single neuron is enough, so both versions still fit the data. On anything
harder, the identical-start network would be stuck at the capacity of one neuron.

### Why some of the original runs overflowed

Several 2024 runs ended in `overflow` warnings and `nan` weights. The cause is the
learning rate. Every gradient picks up a factor of $x$ from each layer, so with inputs
up to $5$ the loss surface is steep, and a step of $\eta = 0.1$ can overshoot the minimum.
Each overshoot then produces a larger gradient and a larger step, until the numbers overflow.
Below, the same network and starting weights (the 2024 values for 3 neurons) are
trained with three learning rates.

In [6]:
def loss_n(a, b, x, y):
    return 0.5 * np.mean((forward_n(a, b, x)[0] - y) ** 2)


a_start = 0.1 + 0.05 * np.arange(3)  # the 2024 starting weights for 3 neurons
b_start = a_start.copy()

for eta in [0.1, 0.03, 0.01]:
    with np.errstate(all="ignore"):  # report the outcome instead of a wall of warnings
        a_end, b_end = train_n(a_start, b_start, x_train, y_train, eta=eta)
        final = loss_n(a_end, b_end, x_train, y_train)
    print(f"eta = {eta:<5} final loss = {final:.3g}")

eta = 0.1   final loss = nan


eta = 0.03  final loss = 2.5e-27
eta = 0.01  final loss = 1.26e-30


## Summary

* I derived backpropagation for a one-hidden-layer leaky-ReLU network by hand, for 2 neurons and then $n$.
* The NumPy implementation matches PyTorch autograd to about $10^{-16}$, which confirms the derivation.
* Two training pitfalls, each shown above: identical initial weights make neurons
  permanently indistinguishable, and a learning rate that is too large for the
  steepness of the loss makes training diverge.

The `classroom/` package in this repository builds on this: it trains whole classrooms
of small networks to compare teaching strategies (see `classroom_experiments.ipynb`).